In [1]:
%pip install ijson

Note: you may need to restart the kernel to use updated packages.


In [2]:
import ijson

count = 0
with open(r"C:\Users\ikhta\OneDrive\Documents\EUR\Year 4\Exchange\BC3415 AI in Accounting & Finance\Practical assignment\Toys_and_Games\Toys_and_Games.jsonl", "rb") as f:
    # 'item' points to each object inside the root-level array
    for item in ijson.items(f, "", multiple_values=True):
        count += 1

print(f"Total observations: {count}")

Total observations: 16260406


In [3]:
import json
import csv

In [5]:
# Configuration
input_file = "C:\\Users\\ikhta\\OneDrive\\Documents\\EUR\\Year 4\\Exchange\\BC3415 AI in Accounting & Finance\\Practical assignment\\Toys_and_Games\\Toys_and_Games.jsonl"
output_file = "balanced_3class_sentiment.csv"
target_per_class = 50000  # 50k negative, 50k neutral, 50k positive

# Trackers
counts = {"negative": 0, "neutral": 0, "positive": 0}


In [6]:

import json
import csv

target_per_class = 50000
counts = {
    "positive": 0,
    "neutral": 0,
    "negative": 0
}

print("Streaming and balancing data into 3 classes...")

with open(input_file, "r", encoding="utf-8") as f_in, \
     open(output_file, "w", newline="", encoding="utf-8") as f_out:

    writer = None

    for line in f_in:
        # Stop when all three classes have reached the target
        if all(c >= target_per_class for c in counts.values()):
            break

        if not line.strip():
            continue

        try:
            review = json.loads(line)

            # Determine sentiment from the rating
            rating = float(review.get("rating", 0))

            if rating in [1.0, 2.0]:
                sentiment = "negative"
            elif rating == 3.0:
                sentiment = "neutral"
            elif rating in [4.0, 5.0]:
                sentiment = "positive"
            else:
                continue

            # Only keep reviews if their class hasn't reached its target
            if counts[sentiment] < target_per_class:

                # Keep every original column and add sentiment
                review["sentiment"] = sentiment

                # Initialise CSV headers from the first selected review
                if writer is None:
                    fieldnames = list(review.keys())
                    writer = csv.DictWriter(
                        f_out,
                        fieldnames=fieldnames,
                        extrasaction="ignore"
                    )
                    writer.writeheader()

                writer.writerow(review)
                counts[sentiment] += 1

        except (json.JSONDecodeError, TypeError, ValueError):
            continue

print("\n--- Sampling Complete ---")
print(f"Saved balanced dataset to: {output_file}")

for sentiment, count in counts.items():
    print(f"  * {sentiment.capitalize()}: {count} reviews")


Streaming and balancing data into 3 classes...

--- Sampling Complete ---
Saved balanced dataset to: balanced_3class_sentiment.csv
  * Positive: 50000 reviews
  * Neutral: 50000 reviews
  * Negative: 50000 reviews


In [7]:
import pandas as pd

In [8]:
reviews_balanced = pd.read_csv("balanced_3class_sentiment.csv")
reviews_balanced.head()

print("\nRating distribution:")
print(
    reviews_balanced["rating"]
    .value_counts(dropna=False)
    .sort_index()
)


Rating distribution:
rating
1.0    30132
2.0    19868
3.0    50000
4.0     8367
5.0    41633
Name: count, dtype: int64


In [10]:
reviews_balanced.shape

(150000, 11)